# FinIA-Flex — Paso 6: Fine-Tuning Ligero (LoRA)

Caso Práctico Unidad 1, materia Generative IA, IEP.

Este paso es para cumplir el requisito 5 del caso (ajustar el LLM en un dataset propio). Uso
los 70 ejemplos que generé por código para ajustar un modelo pequeño con LoRA.

Importante: esto NO le enseña hechos nuevos (eso ya lo hace el RAG) ni a calcular (eso ya
está resuelto en Python). Lo que ajusto es el estilo — que redacte consistente en el formato
de 5 secciones sin que yo tenga que pedírselo con tanto detalle en el prompt.

Uso LoRA en vez de un fine-tuning completo porque ajustar todos los parámetros de un modelo
grande necesita GPUs caras y muchos datos. LoRA solo ajusta una fracción pequeña, así que
corre en la GPU gratuita de Colab (T4) en minutos, con mis 70 ejemplos.

El modelo base es Llama 3.2 1B Instruct (chico, cabe en la T4), y uso Unsloth porque hace el
entrenamiento LoRA más rápido y con menos memoria.


## 1. Configuración del entorno

**Importante:** antes de ejecutar, verificar que el entorno de ejecución de Colab tenga GPU
activada: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.


In [ ]:
!pip install -q unsloth

import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Cargar el modelo base con Unsloth

Se carga en 4-bit para reducir el uso de memoria de la GPU, lo cual es especialmente
importante en la GPU gratuita de Colab.


In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # detección automática según la GPU
    load_in_4bit=True,
)

print("Modelo base cargado.")

## 3. Configurar LoRA

Uso `r=16` y `lora_alpha=16` — son los valores que Unsloth recomienda para este tipo de
ajuste de estilo en modelos chicos. No tiene caso subirle el rango con solo 70 ejemplos, solo
tardaría más sin mejorar nada.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Adaptadores LoRA configurados.")

## 4. Cargar el dataset

Subo `finia_flex_finetuning_dataset.jsonl` (mis 70 ejemplos) al panel de Archivos de Colab, o
lo dejo en mi carpeta de Drive.


In [ ]:
from datasets import load_dataset

# Ajustar la ruta según dónde se haya colocado el archivo
DATASET_PATH = "/content/finia_flex_finetuning_dataset.jsonl"
# Alternativa si se colocó en Google Drive:
# DATASET_PATH = "/content/drive/MyDrive/FinIA-Flex/finia_flex_finetuning_dataset.jsonl"

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Ejemplos cargados: {len(dataset)}")
print(dataset[0])

In [ ]:
def formatear_ejemplo(ejemplo):
    texto = tokenizer.apply_chat_template(
        ejemplo["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": texto}

dataset_formateado = dataset.map(formatear_ejemplo)
print("--- Ejemplo formateado con la plantilla de chat del modelo ---")
print(dataset_formateado[0]["text"][:800])

## 5. Entrenamiento

Con 70 ejemplos y GPU T4, tarda unos 3-8 minutos. Uso pocas épocas (3) porque solo quiero que
aprenda el estilo, no que memorice los ejemplos.

Aquí tuve que resolver dos problemas seguidos: primero usaba `SFTTrainer` de la librería
`trl`, pero esa librería cambia su API muy seguido — me chocó primero con que `tokenizer` ya
se llama `processing_class`, y después con un parámetro interno que ya no existe
(`push_to_hub_token`). Cansado de perseguir esos cambios, cambié a usar el `Trainer` normal de
`transformers` — es una API mucho más estable, y unsloth de todos modos ya me da el modelo
listo con LoRA configurado, así que el entrenamiento en sí es un ciclo estándar sin depender
de `trl` para nada.


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def tokenizar_ejemplo(ejemplo):
    return tokenizer(ejemplo["text"], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)

dataset_tokenizado = dataset_formateado.map(
    tokenizar_ejemplo,
    remove_columns=dataset_formateado.column_names,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    output_dir="/content/finia_flex_lora_output",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenizado,
    data_collator=data_collator,
)

resultado_entrenamiento = trainer.train()
print("Entrenamiento completo.")
print(resultado_entrenamiento)

## 6. Guardar el adaptador LoRA entrenado

Se guarda solo el adaptador (unos pocos megabytes), no el modelo completo — esta es una de
las ventajas de LoRA: el resultado del ajuste es pequeño y portable.


In [ ]:
RUTA_ADAPTADOR = "/content/drive/MyDrive/FinIA-Flex/finia_flex_lora_adapter"

from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained(RUTA_ADAPTADOR)
tokenizer.save_pretrained(RUTA_ADAPTADOR)

print(f"Adaptador LoRA guardado en: {RUTA_ADAPTADOR}")

## 7. Comparación antes / después

Comparo la respuesta del modelo base contra el modelo ya ajustado, con el mismo caso — esta
comparación es mi evidencia principal de este paso.


In [ ]:
CASO_PRUEBA = """DATOS:
Centro de costo: Línea de Producción 2
Categoría: Energía
Presupuesto: $88,000 MXN | Real: $97,500 MXN (Octubre)
Variación: 10.8% ($9,500 MXN)
Histórico: +2.1%, +4.3%
Responsable: Gerente de Línea 2 - M. Torres

INDICADORES CALCULADOS POR EL SISTEMA (usa estos valores tal cual):
- Clasificación de la variación: Variación significativa
- Umbral de aprobación de la categoría "Energía": $30,000 MXN
- ¿La variación de este mes excede el umbral de su categoría?: NO
- Meses consecutivos de sobrecosto (incluyendo el actual): 3
- ¿Aplica la regla de variación sostenida (3+ meses consecutivos de sobrecosto)?: SÍ"""

mensajes_prueba = [
    {"role": "system", "content": (
        "Eres un analista financiero senior de FlexParts Manufacturing MX, especializado en "
        "control de costos de manufactura. Redactas reportes ejecutivos de variación "
        "presupuestal para Gerencia, siguiendo siempre la estructura de 5 secciones: Resumen "
        "Ejecutivo, Diagnóstico por Centro de Costo, Alertas de Política, Recomendación, y "
        "Responsable y Siguiente Paso. Usas únicamente los datos e indicadores que se te "
        "proporcionan, sin inventar cifras, políticas ni responsables."
    )},
    {"role": "user", "content": CASO_PRUEBA},
]

inputs = tokenizer.apply_chat_template(
    mensajes_prueba, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

FastLanguageModel.for_inference(model)  # activa el adaptador LoRA para inferencia rápida
salida_ajustado = model.generate(input_ids=inputs, max_new_tokens=400, temperature=0.3, do_sample=True)
texto_ajustado = tokenizer.decode(salida_ajustado[0][inputs.shape[1]:], skip_special_tokens=True)

print("=" * 90)
print("RESPUESTA DEL MODELO AJUSTADO (con LoRA)")
print("=" * 90)
print(texto_ajustado)

In [ ]:
# Para comparar contra el modelo base, se recarga sin el adaptador LoRA
model_base, tokenizer_base = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model_base)

inputs_base = tokenizer_base.apply_chat_template(
    mensajes_prueba, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

salida_base = model_base.generate(input_ids=inputs_base, max_new_tokens=400, temperature=0.3, do_sample=True)
texto_base = tokenizer_base.decode(salida_base[0][inputs_base.shape[1]:], skip_special_tokens=True)

print("=" * 90)
print("RESPUESTA DEL MODELO BASE (sin ajustar)")
print("=" * 90)
print(texto_base)

## 8. Qué revisar en la comparación

Al comparar las dos salidas de la Sección 7, documentar en el informe:

- ¿El modelo ajustado sigue la estructura de 5 secciones de forma más consistente que el
  modelo base, sin necesidad de que el prompt se lo detalle exhaustivamente?
- ¿El modelo ajustado usa el mismo tono y las mismas frases tipo ("Se activa la regla...",
  "Responsable y Siguiente Paso") que aparecían en el dataset de entrenamiento?
- ¿El modelo base, al no haber visto el formato, tiende a estructurar la respuesta de forma
  más libre o genérica?

Con modelos de 1B de parámetros y un dataset de 70 ejemplos, es normal que la mejora sea
perceptible en **consistencia de formato**, no necesariamente en calidad de razonamiento
financiero — para el razonamiento y los hechos, el prototipo sigue apoyándose en el RAG
(Paso 5) y el cálculo determinístico, no en este modelo ajustado. El fine-tuning aquí cumple
un rol de estilo, no de motor principal de decisión del prototipo.


---
## Resumen (Paso 6)

Ajusté con LoRA (`r=16`) un Llama 3.2 1B en GPU T4 gratuita de Colab, con mis 70 ejemplos
sintéticos que cubren distintos centros de costo, categorías y tipos de alerta.

Decisiones:
- Modelo chico a propósito, para que corra gratis — el modelo principal del prototipo sigue
  siendo Llama 3.3 70B vía Groq (Paso 5), este es solo la demostración de fine-tuning.
- LoRA en vez de fine-tuning completo, por costo y porque solo quiero ajustar estilo.
- 3 épocas, para no sobreajustar con un dataset tan chico.

El hallazgo de este paso ya lo conté arriba (Sección 5): los dos errores de compatibilidad
con `trl`, resueltos cambiando a `Trainer` de `transformers`.

La comparación de la Sección 7 es mi evidencia principal aquí.

Siguiente: Paso 7, el filtro de calidad automático.
